# Session 2 — Working with LLM APIs

**OpenAI · Gemini · Anthropic**

In this notebook we call live LLMs from Python and learn the core building blocks every
LLM app needs: **messages & prompts**, **structured output**, **errors & retries**, and
**tokens & cost**.

> **How to read this notebook:** every code cell has a short note above it. The 🎯 **Purpose**
> line tells you in one glance what the cell does; the text under it explains the code.
>
> **You only need ONE working API key to follow along.** Run the provider sections you have
> keys for and skip the others — the ideas are identical across all three.

## 1. Setup

🎯 **Purpose:** install the three provider SDKs (plus helpers for env vars, schemas).

Run this once. `-q` keeps the output quiet. If a package is already installed, pip just skips it.

In [ ]:
%pip install -q openai google-genai anthropic python-dotenv pydantic

🎯 **Purpose:** load API keys from a `.env` file and confirm which providers are ready.

Create a file named `.env` in this folder with the keys you have:

```
OPENAI_API_KEY=sk-...
GEMINI_API_KEY=...
ANTHROPIC_API_KEY=sk-ant-...
```

`load_dotenv()` reads that file into your environment. The check below prints which keys
are present — it never prints the key values themselves.

In [33]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)  # read .env into environment variables

for key in ["OPENAI_API_KEY", "GEMINI_API_KEY", "ANTHROPIC_API_KEY"]:
    print(f"{key}: {'set' if os.getenv(key) else 'MISSING'}")

OPENAI_API_KEY: set
GEMINI_API_KEY: set
ANTHROPIC_API_KEY: set


🎯 **Purpose:** keep every model ID in ONE place so swapping models is a one-line change.

Model IDs change often. If any call below fails with *"model not found"* (a 404), open the
provider's models page and update the string **here** — nothing else needs to change.

In [34]:
OPENAI_MODEL    = "gpt-5.4-mini"            # a current GPT-5-family model ("*-mini" is cheaper)
GEMINI_MODEL    = "gemini-3.1-flash-lite"   # fast + cheap (newer families also exist)
ANTHROPIC_MODEL = "claude-haiku-4-5"        # flagship ("claude-haiku-4-5" is cheaper/faster)

## 2. Your first LLM call

🎯 **Purpose:** make the simplest possible call to an LLM and print its answer (OpenAI).

The pattern is always: **make a client → send a request → read the text out.** The client
picks up your API key from the environment automatically — never paste keys into code.

💡 **Real-life:** it's like texting a knowledgeable colleague — you send a message, you get a reply. Everything else in this notebook is just controlling *how* that colleague answers.

In [ ]:
from openai import OpenAI

oai = OpenAI()  # reads OPENAI_API_KEY from the environment

resp = oai.responses.create(
    model=OPENAI_MODEL,
    input="Say hello to an AI engineering class in one sentence.",
)
print(resp)

Response(id='resp_0ebd12b304467684006a420cee453c8192aeb15e4963dc63a1', created_at=1782713582.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_0ebd12b304467684006a420ceedd8c819289632df10e7e6b44', content=[ResponseOutputText(annotations=[], text='Hello, AI engineering class—welcome to the exciting world of building intelligent systems!', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=0.98, background=False, completed_at=1782713583.0, conversation=None, max_output_tokens=None, max_tool_calls=None, moderation=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_retention='24h', reasoning=Reasoning(context='current_turn', effort='none', generate_summary=None, summary=None, mode='standard'), safety_identifier=

In [16]:
resp.output_text

'Hello, AI engineering class—welcome to the exciting world of building intelligent systems!'

## 3. The same call on all three providers

🎯 **Purpose:** make the identical request with Gemini — note the different method/field names.

Same three steps as OpenAI; only the SDK's wording changes
(`generate_content` / `contents` / `.text`).

In [ ]:
from google import genai

gem = genai.Client()  # reads GEMINI_API_KEY

resp = gem.models.generate_content(
    model=GEMINI_MODEL,
    contents="Say hello to an AI engineering class in one sentence.",
)
print(resp.text)

🎯 **Purpose:** make the identical request with Anthropic (Claude).

Same idea again. Two Claude-specific things to notice: messages go in a `messages` list of
`{role, content}` items, and `max_tokens` is **required**.

In [24]:
import anthropic

ant = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY

resp = ant.messages.create(
    model=ANTHROPIC_MODEL,
    max_tokens=1024,  # Claude requires a max_tokens limit
    messages=[{"role": "user", "content": "Say hello to an AI engineering class in one sentence."}],
)
print(resp.content[0].text)

Hello to all the brilliant AI engineering students out there—I'm excited to learn and collaborate with you as we explore the fascinating challenges and opportunities in this rapidly evolving field!


### Recap: three SDKs, one idea

| | OpenAI | Gemini | Anthropic |
|---|---|---|---|
| call | `responses.create(input=...)` | `models.generate_content(contents=...)` | `messages.create(messages=[...])` |
| read text | `resp.output_text` | `resp.text` | `resp.content[0].text` |
| `max_tokens` | optional | optional | **required** |

We just wrote the same task three different ways. Hold that thought — it's the reason
frameworks like **LangChain** (next session) exist.

## 4. Messages, roles & system prompts

🎯 **Purpose:** steer the model with a **system prompt** (its persona + rules), shown on all three.

A chat model takes a conversation of **roles**: `system` = who the assistant is and the rules,
`user` = the person, `assistant` = the model's earlier replies. The system prompt is your
main steering wheel.

💡 **Real-life:** think of a **theatre script**. The `system` role is the director's note that defines the character ("you are a calm support agent"); `user` lines are the customer; `assistant` lines are what our character already said. The model simply keeps that character reading its next line.

In [ ]:
SYSTEM = "You are a support assistant for Northstar Services. Answer in ONE sentence."
USER   = "A customer asks how to reset their password."

# OpenAI: the system prompt goes in `instructions`
print("OpenAI   :", oai.responses.create(model=OPENAI_MODEL, instructions=SYSTEM, input=USER).output_text)
# Anthropic: the system prompt is a top-level `system` parameter
print("Anthropic:", ant.messages.create(model=ANTHROPIC_MODEL, max_tokens=200,
                                         system=SYSTEM,
                                         messages=[{"role": "user", "content": USER}]).content[0].text)

# Gemini: the system prompt goes inside a config object
from google.genai import types
print("Gemini   :", gem.models.generate_content(
    model=GEMINI_MODEL, contents=USER,
    config=types.GenerateContentConfig(system_instruction=SYSTEM),
).text)

### When *do* you actually need a system prompt?

A system prompt is **optional** — our first calls had none and still worked. But the moment you build something real, you almost always want one. It is the model's **standing instructions**: its job, personality, rules, and limits. It applies to *every* turn and stays separate from what the end user types.

💡 **Real-life:** a system prompt is the **employee handbook + first-day briefing** you give a new hire *once*, before any customer walks in. The customer's actual question is the **ticket** they bring to the desk. You don't re-explain company policy on every ticket — that's the handbook's (system prompt's) job.

**Reach for a system prompt when you need:**

| Need | Without a system prompt | With one |
|---|---|---|
| **Consistency** across many different user messages | behaviour drifts message to message | same role & rules every time |
| **Control / safety** (rules the user shouldn't override) | the user's text is all there is | "never invent account details; escalate refunds" |
| **A role / persona** | generic assistant | "a Northstar Services support agent" |
| **A fixed output format** | varies | "answer in one sentence" / "return JSON" |
| **Separation of concerns** | rules + data tangled together | system = behaviour · user = just the data |

**Skip it** only for throwaway, one-off questions where the defaults are fine.

> **Rule of thumb:** if you're building a *product*, write a system prompt. The user message should carry only the **data**; the system prompt carries the **behaviour** — so you don't repeat the rules on every request.
>
> ⚠️ A system prompt is a *soft* boundary, not a security wall — a determined user can try to talk around it (*prompt injection*, covered in Session 8). Don't put secrets in it.

### Anatomy of a good system prompt — the common building blocks

Most production system prompts are assembled from the same parts. Use this as a checklist — you rarely need all of them, but decide on each one consciously. (Each block below has a 💡 real-life parallel — picture briefing a brand-new support hire.)

1. **Role / persona** — *who* it is → "You are a support agent for Northstar Services." 💡 *telling the new hire which desk they're on.*
2. **Task / goal** — *what* to do → "Triage the message and draft a first reply." 💡 *their job description.*
3. **Tone / audience** — *how* it sounds → "Friendly, calm, and concise." 💡 *the brand's customer-service voice.*
4. **Rules / guardrails** — the do's and don'ts → "Never invent account details. Escalate refunds to a human." 💡 *the compliance rules every agent must follow.*
5. **Output format** — the shape of the answer → "Reply in 2-3 sentences." / "Return JSON matching the schema." 💡 *"fill in this form," not "write an essay."*
6. **Context / knowledge** — facts to use → business hours, product names, policies. 💡 *the cheat-sheet taped to the desk.*
7. **Fallback behaviour** — what to do when unsure → "If you don't know, say so and offer to escalate — don't guess." 💡 *"if you're not sure, ask your manager — don't make it up."*
8. **(Optional) Examples** — one or two model answers ("few-shot") when the format is tricky. 💡 *showing the new hire a couple of great past replies.*

**Writing tips that actually move the needle:**
- Put the **most important rules first and last** — models weight the edges of the prompt most.
- Be **specific and positive**: "reply in 2-3 sentences" beats "be brief"; *telling it what to do* works better than only *forbidding* things (though "do NOT…" rules still help for hard limits).
- Keep it **stable**: the system prompt is identical on every call. Anything that changes per request (the customer's message, looked-up data) belongs in the **user** message.

### Let's build a system prompt, one upgrade at a time

The best way to *feel* what a system prompt does is to keep the **question fixed** and change **only the prompt** — then watch the answer move. We start with nothing and add **one building block per step**, so the diff is obvious each time.

In [35]:
# The customer's question stays the SAME for every step below.
# We will only ever change the SYSTEM PROMPT and re-run.
USER = "I was charged twice this month. Can I get a refund?"

def ask(system=None):
    """Send USER to Claude with an optional system prompt; return just the reply text."""
    kwargs = dict(model=ANTHROPIC_MODEL, max_tokens=200,
                  messages=[{"role": "user", "content": USER}])
    if system:                       # when system is None we send NO system prompt at all
        kwargs["system"] = system
    return ant.messages.create(**kwargs).content[0].text

#### Step 0 — no system prompt (the baseline)

🎯 **Purpose:** see the model's *default* behaviour with zero steering. Every later step is measured against this answer.

In [36]:
print(ask())   # no system prompt — just the raw question

I'd be happy to help you with this! To look into a duplicate charge, I'll need a bit more information:

1. **What service or company** was the charge from?
2. **When were you charged?** (dates)
3. **What amount** was charged each time?
4. **Do you have your account or order details?** (confirmation numbers, invoice numbers, etc.)

In the meantime, here are some general steps you can take:

- **Check your account** - Log in to see transaction history and confirm it's actually a duplicate
- **Contact the company directly** - Most can quickly review and process refunds for genuine duplicates
- **Gather documentation** - Screenshots or statements showing both charges
- **Check pending transactions** - Sometimes one charge is pending and may still reverse

Once you share more details about which service was charged, I can give you more specific guidance on how to get this resolved


#### Step 1 — add a role + tone  ➕ *who it is, how it sounds*

🎯 **Purpose:** the two most basic blocks — **role** (#1) and **tone** (#3).

**What's new vs Step 0:** we say it's a *Northstar support agent* and to be *warm and concise*.  
**Watch:** the reply picks up a consistent voice instead of a generic one.

In [28]:
SYSTEM = "You are a friendly customer-support agent for Northstar Services. Be warm, clear, and concise."

print(ask(SYSTEM))

I'm sorry to hear you were charged twice—that's definitely frustrating! I'm happy to help you get this sorted out.

To look into your account and process a refund, I'll need a bit of information:

1. **Your account number** or the **email address** associated with your account
2. **Approximate dates** of the duplicate charges
3. **The amount** charged each time

Once I have these details, I can investigate what happened and get that resolved for you right away.

Is there anything else I can help clarify in the meantime?


#### Step 2 — add an output-format rule  ➕ *control the shape*

🎯 **Purpose:** add block **#5, output format**.

**What's new vs Step 1:** one extra line — *reply in no more than two sentences.*  
**Watch:** same voice as Step 1, but now the length is under your control.

In [29]:
SYSTEM = (
    "You are a friendly customer-support agent for Northstar Services. Be warm, clear, and concise.\n"
    "Reply in no more than TWO sentences."        # <-- the only new line vs Step 1
)

print(ask(SYSTEM))

I'm sorry to hear you were charged twice! I'd be happy to help resolve this—could you please provide your account number or the email address associated with your account so I can look into the duplicate charges?


#### Step 3 — add guardrails + a fallback  ➕ *the rules it must not break*  ⭐

🎯 **Purpose:** the blocks that make it *safe to ship* — **rules/guardrails** (#4) and **fallback when unsure** (#7).

**What's new vs Step 2:** explicit *do-not* rules and an escalation path.  
**Watch — this is the payoff:** Step 0 probably *offered a refund*; this version **refuses to decide and escalates to a human** instead. That exact behaviour is the seed of the project's `draft_reply()`.

In [30]:
SYSTEM = (
    "You are a support agent for Northstar Services, a SaaS company.\n"
    "- Be friendly, calm, and concise (2-3 sentences).\n"
    "- NEVER invent account details, charges, or refund decisions.\n"          # <-- guardrail
    "- Refunds and billing disputes MUST be escalated to a human - say so.\n"  # <-- guardrail
    "- If you are unsure, offer to connect them with a teammate rather than guess."  # <-- fallback
)

print(ask(SYSTEM))

I'm sorry to hear you were charged twice—that's frustrating! I'm not able to process refunds directly, but this is definitely something we can fix. Let me connect you with our billing team who can review your account and process a refund right away.


#### What changed, step by step

| Step | We added | Checklist block | Effect on the answer |
|---|---|---|---|
| 0 | nothing | — | generic; may over-promise (offers a refund) |
| 1 | role + tone | 1, 3 | a consistent Northstar voice |
| 2 | + format rule | 5 | length under control |
| 3 | + guardrails + fallback | 4, 7 | refuses risky promises, escalates safely |

Same question every time — the *only* thing that changed was the system prompt. That is your steering wheel.

🎯 **Purpose:** show multi-turn chat — the model has no memory, so we resend the history.

The API is **stateless**: "memory" just means sending the prior turns again each call.
Anthropic: you keep the list yourself. Gemini: a `chat` object keeps it for you (a convenience
wrapper around the same idea).

💡 **Real-life:** the model has **amnesia between calls** (think the film *Memento*, or a doctor who has never met you before). To continue a conversation you hand it the whole file again every time — the `chat` object just does that re-filing for you automatically.

In [ ]:
# Anthropic — you build the running conversation list yourself
history = [
    {"role": "user",      "content": "My invoice looks wrong."},
    {"role": "assistant", "content": "I can help. What's the invoice number?"},
    {"role": "user",      "content": "It's INV-4821."},
]
print("Anthropic:", ant.messages.create(model=ANTHROPIC_MODEL, max_tokens=200, messages=history).content[0].text)

# Gemini — a chat object tracks the history for you
chat = gem.chats.create(model=GEMINI_MODEL)
print("Gemini 1 :", chat.send_message("My invoice looks wrong.").text)
print("Gemini 2 :", chat.send_message("It's INV-4821.").text)  # remembers the prior turn

## 5. Structured output with Pydantic

🎯 **Purpose:** get **typed, validated data** back from the model instead of free text.

So far the model has handed us **prose** — perfect for a human to read, but awkward for code to use. Structured output flips that: we hand the model a **form** (a Pydantic model) and it fills in the boxes.

💡 **Real-life:** it's the difference between a customer **writing you a paragraph** ("I was charged twice and I'm furious!") and you transferring it onto a **support-ticket form** — *Category: Billing · Urgency: High · Sentiment: Negative*. The paragraph is for reading; the form is for routing, counting, and automating.

### When *do* you need structured output?

The rule of thumb: **the moment another piece of code has to act on the answer, you want fields, not prose.**

| You need to… | With plain text | With structured output |
|---|---|---|
| **branch in code** | parse the prose yourself (brittle) | `if t.category == "billing": ...` |
| **store it** (DB / sheet) | messy, inconsistent | each field is a column |
| **count / analyse** | nearly impossible | group by `category`, `urgency` |
| **feed the next step** | fragile string-handling | pass a typed object along |
| **guarantee valid values** | the model may say anything | `Literal` enforces the menu |

**Skip it** when the deliverable *is* the prose — a drafted reply, a summary meant to be read. (That's exactly why our project uses *structured* output for triage, but *plain* text for the draft reply.)

### Anatomy of a good schema — the building blocks

A Pydantic model is just a **form definition**. The common pieces (each with its 💡 real-life parallel):

1. **Fields** — the boxes on the form → `category`, `summary`. 💡 *columns in a spreadsheet.*
2. **Types** — what each box holds → `str`, `int`, `bool`. 💡 *a number box vs a date box.*
3. **`Literal` / enum** — a fixed menu of allowed values → `Literal["low", "medium", "high"]`. 💡 *a dropdown, not a blank.*
4. **`Field(description=...)`** — a hint written next to the box → guides the model. 💡 *the "MM/DD/YYYY" under a date field.*
5. **Optional / defaults** — boxes allowed to be blank → `tags: list[str] = []`. 💡 *the "optional" fields on a form.*
6. **Nested models / lists** — a form inside a form → `items: list[LineItem]`. 💡 *the line items on an invoice.*

**Tips:** name fields like database columns (clear, consistent); reach for `Literal` whenever the value should be one of a known set — it turns *"please classify"* into a **guarantee**; and add a `description` to anything ambiguous — it's a mini system prompt for that one field.

### Let's build the schema, one upgrade at a time

Just like the system prompt, we keep the **message fixed** and grow the **schema** one step at a time — watching the output get more structured and more reliable at each step.

In [52]:
from pydantic import BaseModel, Field
from typing import Literal

# One fixed customer message for every step; only the SCHEMA will change.
MESSAGE = "I've been charged twice for May and nobody has replied to my emails. This is ridiculous."

def extractOpenAI(schema):
    """Ask OpenAI to fill in `schema` from MESSAGE; return the validated object."""
    response = oai.responses.parse(
        model=OPENAI_MODEL,
        input=MESSAGE,
        text_format=schema
    )
    return response.output_parsed

def extractAntropic(schema):
    """Ask Antropic to fill in `schema` from MESSAGE; return the validated object."""
    response = ant.messages.parse(
        model=ANTHROPIC_MODEL,
        max_tokens=1024,
        messages=[{"role": "user", "content": MESSAGE}],
        output_format=schema
    )
    return response.parsed_output

def extractGemini(schema):
    """Ask Gemini to fill in `schema` from MESSAGE; return the validated object."""
    response = gem.interactions.create(
        model="gemini-3.5-flash",
        input=MESSAGE,
        response_format={
            "type": "text",
            "mime_type": "application/json",
            "schema": schema.model_json_schema()
        },
    )
    output = schema.model_validate_json(response.output_text)
    return output

#### Step 0 — the simplest possible schema (one field)

🎯 **Purpose:** prove the basic move — define a form with one box, get a typed object back.

💡 **Real-life:** even a one-line form ("Summary: ______") is already easier to file than a paragraph.

In [53]:
class TriageV0(BaseModel):
    summary: str          # a single free-text box

r1 = extractOpenAI(TriageV0)
r2 = extractAntropic(TriageV0)
r3 = extractGemini(TriageV0)
print("OpenAI Structured Response:\n", r1)
print("Antropic Structured Response:\n", r2)
print("Gemini Structured Response:\n", r3)

OpenAI Structured Response:
 summary='The user reports being charged twice for May and has not received replies to their emails. They are frustrated and want a resolution.'
Antropic Structured Response:
 summary='Customer reports duplicate charges for May and unresponsive support team via email'
Gemini Structured Response:
 summary='The customer is complaining about being charged twice for the month of May and is frustrated by the lack of response to their emails.'


#### Step 1 — add another field, but as plain `str`  ➕ *more boxes*

🎯 **Purpose:** add a `category` — and see the problem with leaving it free-text.

**Watch:** the model picks *some* category, but the wording is unpredictable — `"Billing"`, `"billing issue"`, `"Double charge"`. That's hard to switch on in code.

In [55]:
class TriageV1(BaseModel):
    category: str         # plain string -> the model can return ANYTHING here
    summary:  str

t = extractGemini(TriageV1)
print(t)
print("Unpredictable value ->", repr(t.category))   # could be any wording

category='Billing' summary='Charged twice for May and received no response to support emails.'
Unpredictable value -> 'Billing'


#### Step 2 — constrain it with `Literal`  ➕ *a dropdown instead of a blank*  ⭐

🎯 **Purpose:** turn `category` into a **fixed menu** with `Literal`.

**What's new vs Step 1:** `category` can now only be one of four values.
**Watch:** the output is guaranteed to be one of the allowed options — safe to `if` / `switch` on.

💡 **Real-life:** you swapped a blank text box for a **dropdown menu** — no more "Billing" vs "billing issue" surprises.

In [56]:
class TriageV2(BaseModel):
    category: Literal["billing", "technical", "account", "general"]   # a fixed menu
    summary:  str

t = extractGemini(TriageV2)
print(t)
print("Guaranteed one of the four ->", t.category)

category='billing' summary='User was charged twice for May and has not received any response to support emails.'
Guaranteed one of the four -> billing


#### Step 3 — add the rest + label the boxes with `Field(...)`  ➕ *descriptions guide the model*

🎯 **Purpose:** add `urgency` and `sentiment` (more menus) and a **`Field(description=...)`** for the summary.

**What's new vs Step 2:** two more `Literal` fields, plus a description telling the model exactly what `summary` should contain.
**Watch:** the description acts like a mini-instruction for that single field. This is the exact `Triage` model our project uses.

In [57]:
class Triage(BaseModel):
    category:  Literal["billing", "technical", "account", "general"]
    urgency:   Literal["low", "medium", "high"]
    sentiment: Literal["negative", "neutral", "positive"]
    summary:   str = Field(description="One-line summary of what the customer wants")  # <-- labelled box

t = extractGemini(Triage)
print(t)
print("\nSwitch on any field directly ->", t.category, "|", t.urgency, "|", t.sentiment)

category='billing' urgency='high' sentiment='negative' summary='Customer was double-charged for May and has not received any replies to their emails.'

Switch on any field directly -> billing | high | negative


#### What changed, step by step

| Step | We added | Building block | Why it matters |
|---|---|---|---|
| 0 | `summary: str` | a field | a typed object instead of prose |
| 1 | `category: str` | another field | works, but values are unpredictable |
| 2 | `category: Literal[...]` | enum / dropdown | values are now **guaranteed** |
| 3 | + `urgency`, `sentiment`, `Field(description=...)` | menus + labels | the full, production-ready `Triage` |

Same message every time — the richer the schema, the more reliable and usable the data.

## 6. Errors, retries, tokens & cost

🎯 **Purpose:** handle failures gracefully (rate limits, network issues).

Real calls fail sometimes. The SDKs already **retry** transient errors (429 / 5xx) with backoff;
your job is to catch what's left and decide. (Anthropic and Gemini have the same idea — see the
comment for their exception names.)

💡 **Real-life:** a **429 rate-limit** is the *"all our agents are busy, please hold"* message — the fix is to wait and redial, which the SDK does for you. A **5xx** is the provider's switchboard glitching; a **connection error** is your own Wi-Fi dropping.

In [64]:
import openai

oai = OpenAI(max_retries=3, timeout=30)  # SDK auto-retries 429 / 5xx with backoff

try:
    resp = oai.responses.create(model=OPENAI_MODEL, input="hi")
    print(resp.output_text)
except openai.RateLimitError:
    print("Rate limited - slow down / retry later")
except openai.APIStatusError as e:
    print("API error:", e.status_code)
except openai.APIConnectionError:
    print("Network problem")

# Anthropic: anthropic.RateLimitError / anthropic.APIStatusError / anthropic.APIConnectionError
# Gemini:    from google.genai import errors  ->  except errors.APIError as e: print(e.code, e.message)

Hi! How can I help?


🎯 **Purpose:** read how many **tokens** each call used (every response reports this).

Tokens are the unit you pay for. Each provider exposes input vs output token counts in a
slightly different place — shown below.

💡 **Real-life:** a token is a chunk of text (~¾ of a word, so ~750 words ≈ 1,000 tokens). Think of it like a **taxi meter** or an old **pay-by-the-word telegram** — the longer the prompt *and* the reply, the more it ticks.

In [65]:
r_oai = oai.responses.create(model=OPENAI_MODEL, input=MESSAGE)
print("OpenAI   :", r_oai.usage.input_tokens, r_oai.usage.output_tokens, r_oai.usage.total_tokens)

r_ant = ant.messages.create(model=ANTHROPIC_MODEL, max_tokens=200,
                            messages=[{"role": "user", "content": MESSAGE}])
print("Anthropic:", r_ant.usage.input_tokens, r_ant.usage.output_tokens)

r_gem = gem.models.generate_content(model=GEMINI_MODEL, contents=MESSAGE)
um = r_gem.usage_metadata
print("Gemini   :", um.prompt_token_count, um.candidates_token_count, um.total_token_count)

OpenAI   : 24 107 131
Anthropic: 27 200
Gemini   : 21 523 544


🎯 **Purpose:** turn tokens into money.

Cost = tokens × price-per-million. Output tokens are usually pricier than input. Always check
the provider's current pricing page — the numbers below are just an example.

💡 **Real-life:** **output costs more than input** — like a consultant who bills more for the hours they *talk* than the hours they *read* your brief. Long answers are where the bill grows, so "be concise" saves money as well as time.

In [ ]:
def cost(in_tok, out_tok, in_price_per_mtok, out_price_per_mtok):
    return in_tok / 1e6 * in_price_per_mtok + out_tok / 1e6 * out_price_per_mtok

$ 0.01475


## 7. Wrap-up

We called three providers, steered them with system prompts, got **validated structured data**,
handled errors, and measured cost.

But notice: to support all three providers we wrote the same logic three times, with three
SDKs, three ways to read the text, and three error types. **Next session we fix that** with
**LangChain** — one interface across providers, plus the building blocks (tools, memory,
structured output) we'll need as the project grows.

➡️ Now open **`main.py`** (in the project root) to start building the course-long project.